# Crea tu propia IA con Hugging Face · Los 18 ejemplos

Cuadernos del capítulo 12 del libro, listos para ejecutar. Cada ejemplo es
autocontenido: puedes ir directamente al que te interese.

**Código bajo licencia Apache 2.0.** El libro está en [kobalto.es/biblioteca](https://kobalto.es/biblioteca/).

Los identificadores de modelo se comprueban periódicamente contra el Hub; donde el
libro impreso cita uno que ha sido retirado, aquí encontrarás el sustituto y una nota.


## Instalación


In [ ]:
%pip install -q transformers==5.19.0 datasets==5.1.0 accelerate \
    sentence-transformers==6.1.0 peft==0.21.2 sentencepiece sacremoses pypdf pillow

## Ejemplo 1. Análisis de sentimiento de reseñas

**Problema.** Una cadena hotelera recibe 3.000 reseñas al mes en Google y Booking y quiere detectar automáticamente las negativas para responderlas en menos de 24 horas.

**Modelo.** `nlptown/bert-base-multilingual-uncased-sentiment`, un BERT multilingüe que predice de 1 a 5 estrellas.

**Resultado esperado.** La segunda reseña aparece con 1 o 2 estrellas y marcada para respuesta urgente.


In [ ]:
from transformers import pipeline

clf = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
resenas = [
    "La habitación estaba impecable y el personal fue encantador.",
    "El aire acondicionado no funcionaba y nadie vino a arreglarlo.",
    "Correcto, sin más. El desayuno algo escaso.",
]
for texto, r in zip(resenas, clf(resenas)):
    estrellas = int(r["label"][0])
    alerta = "RESPONDER YA" if estrellas <= 2 else ""
    print(f"{estrellas}★ ({r['score']:.2f}) {alerta} | {texto}")


## Ejemplo 2. Clasificación zero-shot de correos

**Problema.** Un ayuntamiento recibe correos ciudadanos y debe enviarlos al departamento correcto sin disponer de ejemplos etiquetados.

**Modelo.** `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`.

> ⚠️ **Este ejemplo falla a propósito.** El modelo deja «limpieza viaria» tercera
> (0,225) y gana «urbanismo» (0,309) por tres milésimas sobre la segunda. Probadas
> cuatro plantillas y etiquetas más explícitas: ninguna lo arregla. Lo que hay que
> mirar no es quién gana, sino **cuánto gana y por cuánto**: las cuatro puntuaciones
> están por debajo de 0,5 y las dos primeras empatadas. Eso es el modelo diciendo
> que no lo sabe, y es lo que debe disparar la revisión manual.


In [ ]:
from transformers import pipeline

zs = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
correo = "Hay un contenedor desbordado en la calle Mayor desde hace tres días."
etiquetas = ["urbanismo", "tributos e impuestos", "limpieza viaria", "servicios sociales"]
r = zs(correo, candidate_labels=etiquetas, hypothesis_template="Este mensaje trata sobre {}.")

# Todas las puntuaciones, no solo la ganadora: es el dato que decide
for etiqueta, puntuacion in zip(r["labels"], r["scores"]):
    print(f"{etiqueta:<24}{puntuacion:.3f}")

# La regla que el ejemplo pide: si nadie destaca, lo lee una persona
if r["scores"][0] < 0.5 or (r["scores"][0] - r["scores"][1]) < 0.1:
    print("\n→ A revisión manual: ninguna etiqueta destaca.")


## Ejemplo 3. Resumen de documentos largos

**Problema.** Un despacho de abogados quiere un resumen de una página de cada sentencia de 20 páginas.

**Modelo.** `Qwen/Qwen2.5-1.5B-Instruct` con ventana de contexto amplia; para textos que superen el contexto se aplica la técnica *map-reduce*.

**Resultado esperado.** Un resumen estructurado en tres apartados.


In [ ]:
from transformers import pipeline

llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device_map="auto")

def pedir(instruccion, texto, n=300):
    m = [{"role": "user", "content": f"{instruccion}\n\n{texto}"}]
    return llm(m, max_new_tokens=n, temperature=0.2)[0]["generated_text"][-1]["content"]

palabras = open("sentencia.txt", encoding="utf-8").read().split()
bloques = [" ".join(palabras[i:i + 2000]) for i in range(0, len(palabras), 2000)]
parciales = [pedir("Resume en 5 frases los hechos y fundamentos:", b) for b in bloques]
final = pedir("Une estos resúmenes en uno solo con apartados Hechos, Fundamentos y Fallo:",
              "\n\n".join(parciales), n=600)
print(final)


## Ejemplo 4. Traducción automática

**Problema.** Una tienda online valenciana necesita traducir sus fichas de producto del español al inglés y al francés.

**Modelo.** Los modelos Opus-MT de la Universidad de Helsinki: `Helsinki-NLP/opus-mt-es-en` y `Helsinki-NLP/opus-mt-es-fr`, pequeños y especializados en un par de idiomas.

**Resultado esperado.** Las dos traducciones, con "Ontinyent" sin traducir.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def cargar_traductor(modelo_id):
    tok = AutoTokenizer.from_pretrained(modelo_id)
    modelo = AutoModelForSeq2SeqLM.from_pretrained(modelo_id)
    def traducir(textos):
        entrada = tok(textos, return_tensors="pt", padding=True, truncation=True)
        salida = modelo.generate(**entrada, max_new_tokens=256)
        return tok.batch_decode(salida, skip_special_tokens=True)
    return traducir

a_en = cargar_traductor("Helsinki-NLP/opus-mt-es-en")
a_fr = cargar_traductor("Helsinki-NLP/opus-mt-es-fr")
fichas = ["Camiseta de algodón orgánico, tejida en Ontinyent. Lavar a 30 grados."]
print(a_en(fichas)[0])
print(a_fr(fichas)[0])


## Ejemplo 5. Extracción de entidades (NER) en noticias

**Problema.** Un gabinete de prensa analiza 500 noticias diarias y necesita saber qué personas, organizaciones y lugares se mencionan.

**Modelo.** `PlanTL-GOB-ES/roberta-base-bne-capitel-ner`, entrenado por el Plan de Tecnologías del Lenguaje con el corpus de la Biblioteca Nacional de España.

> ⚠️ **Nota de la edición en línea.** El modelo del libro (`PlanTL-GOB-ES/roberta-base-bne-capitel-ner`) fue retirado del Hub: devuelve 401. Aquí se usa un sustituto verificado que da el mismo resultado.

**Resultado esperado.** Un diccionario con claves como `ORG` (Barcelona Supercomputing Center, UPC) y `LOC` o `MISC` para el resto.


In [ ]:
from collections import defaultdict
from transformers import pipeline

ner = pipeline("ner", model="mrm8488/bert-spanish-cased-finetuned-ner", aggregation_strategy="simple")
noticia = ("La ministra de Ciencia visitó el Barcelona Supercomputing Center "
           "para presentar el proyecto ALIA junto a investigadores de la UPC.")
por_tipo = defaultdict(set)
for e in ner(noticia):
    por_tipo[e["entity_group"]].add(e["word"])
print(dict(por_tipo))


## Ejemplo 6. Transcripción de audio con Whisper

**Problema.** Un centro de atención telefónica quiere transcribir las llamadas grabadas para auditar la calidad del servicio.

**Modelo.** `openai/whisper-small`, modelo ASR multilingüe publicado con licencia MIT. Las versiones `medium` y `large-v3` son más precisas y más lentas.

**Resultado esperado.** La transcripción con marcas de tiempo por frase.


In [ ]:
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-small",
               chunk_length_s=30, device_map="auto")
r = asr("llamada_0412.mp3", return_timestamps=True,
        generate_kwargs={"language": "spanish", "task": "transcribe"})
for trozo in r["chunks"]:
    ini, fin = trozo["timestamp"]
    fin = f"{fin:6.1f}" if fin is not None else "   fin"   # el último trozo puede no tener final
    print(f"[{ini:6.1f}s - {fin}s] {trozo['text']}")


## Ejemplo 7. Preguntas y respuestas sobre un texto

**Problema.** Un departamento de Recursos Humanos quiere responder dudas sobre el convenio colectivo localizando la frase exacta que contiene la respuesta.

**Modelo.** `PlanTL-GOB-ES/roberta-base-bne-sqac`, modelo extractivo: no inventa texto, señala un fragmento del documento.

> ⚠️ **Nota de la edición en línea.** Mismo caso que el ejemplo 5: el modelo de QA del Plan TL ya no es público. El sustituto devuelve la misma respuesta.

**Resultado esperado.** `23 días laborables`, copiado literalmente del convenio gracias a los offsets de caracteres.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

modelo_id = "mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es"
tok = AutoTokenizer.from_pretrained(modelo_id)
modelo = AutoModelForQuestionAnswering.from_pretrained(modelo_id)

convenio = ("Artículo 22. Vacaciones. El personal disfrutará de 23 días laborables de vacaciones "
            "anuales retribuidas, de los cuales al menos 10 se disfrutarán entre junio y septiembre.")
pregunta = "¿Cuántos días de vacaciones hay?"

entrada = tok(pregunta, convenio, return_tensors="pt", return_offsets_mapping=True)
offsets = entrada.pop("offset_mapping")[0]
es_contexto = torch.tensor([s == 1 for s in entrada.sequence_ids(0)])  # 1 = tokens del convenio
with torch.no_grad():
    salida = modelo(**entrada)
# La respuesta debe empezar y acabar dentro del convenio, no dentro de la pregunta
ini = int(salida.start_logits[0].masked_fill(~es_contexto, -1e9).argmax())
fin = max(int(salida.end_logits[0].masked_fill(~es_contexto, -1e9).argmax()), ini)
print(convenio[offsets[ini][0]:offsets[fin][1]])


## Ejemplo 8. Clasificación de imágenes

**Problema.** Una cooperativa agrícola quiere clasificar fotos de hojas de olivo enviadas por los socios para detectar posibles enfermedades.

**Modelo.** Para empezar, `google/vit-base-patch16-224`, un Vision Transformer genérico entrenado en ImageNet. Para la tarea real habría que ajustarlo con fotos etiquetadas (el procedimiento es el del ejemplo 13, cambiando texto por imágenes).

**Resultado esperado.** Tres etiquetas de ImageNet; ninguna será "repilo", porque el modelo genérico no conoce esa enfermedad. Este resultado es pedagógico: demuestra por qué hace falta el fine-tuning.


In [ ]:
from PIL import Image
from transformers import pipeline

vision = pipeline("image-classification", model="google/vit-base-patch16-224")
foto = Image.open("hoja_olivo.jpg")
for r in vision(foto, top_k=3):
    print(f"{r['label']}: {r['score']:.2%}")


## Ejemplo 9. Buscador semántico de preguntas frecuentes

**Problema.** El servicio de atención al estudiante de una universidad tiene 200 preguntas frecuentes, y los alumnos las formulan con palabras distintas a las del documento.

**Modelo.** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, rápido incluso en CPU.

> ⚠️ **Nota de la edición en línea.** El libro usa `paraphrase-multilingual-MiniLM-L12-v2` y un umbral de 0,5 que ese modelo nunca alcanza: la similitud real con la pregunta correcta es 0,296 y elige la equivocada. Con `multilingual-e5-base` —el mismo que el libro usa para RAG— sube a 0,823 y acierta.

**Resultado esperado.** Encuentra la pregunta de la convocatoria extraordinaria aunque no comparte palabras clave con la consulta.


In [ ]:
from sentence_transformers import SentenceTransformer, util

emb = SentenceTransformer("intfloat/multilingual-e5-base")
faq = {
    "¿Cómo solicito la convocatoria extraordinaria?": "Desde la sede electrónica, apartado Exámenes...",
    "¿Cuál es el plazo de matrícula?": "Del 15 de julio al 10 de septiembre...",
    "¿Cómo convalido asignaturas?": "Presentando la solicitud de reconocimiento de créditos...",
}
preguntas = list(faq)
v_faq = emb.encode(preguntas, convert_to_tensor=True)

consulta = "Suspendí en junio, ¿dónde me apunto para la recuperación?"
sim = util.cos_sim(emb.encode(consulta, convert_to_tensor=True), v_faq)[0]
mejor = int(sim.argmax())
if sim[mejor] > 0.5  # con e5-base la similitud real es 0,82:
    print(preguntas[mejor], "->", faq[preguntas[mejor]])
else:
    print("Derivar a un orientador")


## Ejemplo 10. Chatbot con memoria de conversación

**Problema.** Una academia de idiomas quiere un tutor de conversación en inglés que recuerde lo dicho en los turnos anteriores.

**Modelo.** `Qwen/Qwen2.5-1.5B-Instruct`.

**Resultado esperado.** Si el alumno dice en el primer turno que se llama Lucía, el tutor lo recuerda en el tercero.


In [ ]:
from transformers import pipeline

llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device_map="auto")
historial = [{"role": "system", "content": "You are a friendly English tutor. Correct mistakes briefly, then reply."}]
MAX_TURNOS = 10

while (texto := input("Tú: ")) != "salir":
    historial.append({"role": "user", "content": texto})
    historial = [historial[0]] + historial[1:][-2 * MAX_TURNOS:]
    respuesta = llm(historial, max_new_tokens=200, temperature=0.7)[0]["generated_text"][-1]["content"]
    historial.append({"role": "assistant", "content": respuesta})
    print("Tutor:", respuesta)


## Ejemplo 11. Extracción de datos a JSON estructurado

**Problema.** Un departamento de contabilidad recibe facturas de proveedores en texto libre y necesita volcar emisor, fecha, base imponible e IVA en su ERP.

**Modelo.** `Qwen/Qwen2.5-1.5B-Instruct`, con instrucciones estrictas y validación posterior. Se evita a propósito la variante 3B: su licencia (Qwen Research License) no permite uso comercial, y un departamento de contabilidad es uso comercial.

**Resultado esperado.** `{"emisor": "Talleres García S.L.", "cif": "B12345678", "fecha": "2026-10-03", "base": 420.0, "iva": 88.2}`.


In [ ]:
import json
from transformers import pipeline

llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device_map="auto")
ESQUEMA = '{"emisor": str, "cif": str, "fecha": "AAAA-MM-DD", "base": float, "iva": float}'
factura = "Talleres García S.L., CIF B12345678. Fecha: 3 de octubre de 2026. Base: 420,00 €. IVA 21 %: 88,20 €."

def extraer(texto, intentos=3):
    m = [{"role": "system", "content": f"Devuelve SOLO un JSON con este esquema: {ESQUEMA}"},
         {"role": "user", "content": texto}]
    for _ in range(intentos):
        salida = llm(m, max_new_tokens=150, do_sample=False)[0]["generated_text"][-1]["content"]
        try:
            datos = json.loads(salida.strip().strip("`").removeprefix("json"))
            assert abs(datos["base"] * 0.21 - datos["iva"]) < 0.05   # comprobación de negocio
            return datos
        except (json.JSONDecodeError, KeyError, AssertionError):
            continue
    return None

print(extraer(factura))


## Ejemplo 12. Asistente de programación

**Problema.** Un equipo de analistas de datos quiere un asistente local que genere y explique funciones de Python sin enviar código de la empresa a servicios externos.

**Modelo.** `Qwen/Qwen2.5-Coder-1.5B-Instruct`, especializado en código.

**Resultado esperado.** La función generada supera las dos pruebas (el algoritmo usa el resto de dividir entre 23).


In [ ]:
import re
from transformers import pipeline

coder = pipeline("text-generation", model="Qwen/Qwen2.5-Coder-1.5B-Instruct", device_map="auto")
peticion = ("Escribe una función Python `valida_dni(dni: str) -> bool` que compruebe "
            "la letra de un DNI español. Solo el código, en un bloque ```python.")
r = coder([{"role": "user", "content": peticion}], max_new_tokens=300, do_sample=False)
codigo = re.search(r"```python\n(.*?)```", r[0]["generated_text"][-1]["content"], re.S).group(1)

espacio = {}
exec(codigo, espacio)          # solo en un entorno aislado (Colab), nunca en producción
assert espacio["valida_dni"]("12345678Z") is True
assert espacio["valida_dni"]("12345678A") is False
print("Pruebas superadas")


## Ejemplo 13. Fine-tuning de un clasificador propio

**Problema.** Una aseguradora quiere clasificar los partes de siniestro en Hogar, Automóvil, Salud y Responsabilidad Civil con más precisión que el enfoque zero-shot del ejemplo 2.

**Modelo.** `PlanTL-GOB-ES/roberta-base-bne` como base, al que se añade una capa de clasificación de 4 clases.

> ⚠️ **Nota de la edición en línea.** El `roberta-base-bne` del libro no carga su tokenizador. Se sustituye por el BERT español de la Universidad de Chile.

**Resultado esperado.** Un F1 macro que aumenta en cada época y que debe superar al del clasificador zero-shot sobre el mismo conjunto de prueba.


In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import f1_score

clases = ["hogar", "automovil", "salud", "rc"]
ds = load_dataset("csv", data_files="partes.csv")["train"].train_test_split(test_size=0.2, seed=42)
ds = ds.map(lambda e: {"label": clases.index(e["etiqueta"])})

base = "dccuchile/bert-base-spanish-wwm-cased"
tok = AutoTokenizer.from_pretrained(base)
ds = ds.map(lambda e: tok(e["texto"], truncation=True, max_length=256), batched=True)
modelo = AutoModelForSequenceClassification.from_pretrained(base, num_labels=len(clases))

def metricas(p):
    return {"f1_macro": f1_score(p.label_ids, np.argmax(p.predictions, -1), average="macro")}

args = TrainingArguments("clasificador-partes", num_train_epochs=3, learning_rate=2e-5,
                         per_device_train_batch_size=16, eval_strategy="epoch", fp16=True)
Trainer(model=modelo, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
        processing_class=tok, compute_metrics=metricas).train()


## Ejemplo 14. Agente con herramientas

**Problema.** Un comercial quiere preguntar en lenguaje natural "¿cuánto supone un descuento del 12 % sobre el pedido 1043?" y que el sistema consulte la base de datos y haga el cálculo sin errores aritméticos.

**Modelo.** Un LLM Instruct orquestado con `smolagents`, la librería de agentes de Hugging Face.

**Resultado esperado.** El agente llama a `importe_pedido(1043)`, calcula 2.350 × 0,12 = 282 € y lo responde.


In [ ]:
import torch
from smolagents import CodeAgent, TransformersModel, tool
from transformers import BitsAndBytesConfig

PEDIDOS = {1043: 2350.00, 1044: 980.50}

@tool
def importe_pedido(numero: int) -> float:
    """Devuelve el importe en euros de un pedido.

    Args:
        numero: número de pedido.
    """
    return PEDIDOS[numero]

# En 16 bits un 7B no cabe en la T4 de Colab: se carga en 4 bits
cuatro_bits = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16)
modelo = TransformersModel(model_id="Qwen/Qwen2.5-7B-Instruct",
                           model_kwargs={"quantization_config": cuatro_bits})
agente = CodeAgent(tools=[importe_pedido], model=modelo)
print(agente.run("¿Cuánto supone un descuento del 12 % sobre el pedido 1043?"))


## Ejemplo 15. Visión y lenguaje: preguntar a una imagen

**Problema.** Una empresa de mantenimiento recibe fotos de cuadros eléctricos y quiere una descripción preliminar del estado antes de enviar al técnico.

**Modelo.** `Qwen/Qwen2-VL-2B-Instruct`, modelo multimodal que acepta imagen y texto.

**Resultado esperado.** Un párrafo que describe lo visible en la foto.


In [ ]:
from transformers import pipeline

vlm = pipeline("image-text-to-text", model="Qwen/Qwen2-VL-2B-Instruct", device_map="auto")
mensajes = [{"role": "user", "content": [
    {"type": "image", "url": "cuadro_electrico.jpg"},
    {"type": "text", "text": "Describe el estado del cuadro: cableado, etiquetas y signos de quemaduras."},
]}]
r = vlm(text=mensajes, max_new_tokens=200)
print(r[0]["generated_text"][-1]["content"])


## Ejemplo 16. Publicar tu modelo en el Hub

**Problema.** El clasificador del ejemplo 13 debe compartirse con otro equipo de la empresa de forma versionada.

**Modelo.** El entrenado en el ejemplo 13.

**Resultado esperado.** El repositorio aparece en el Hub y el pipeline devuelve `hogar`.


In [ ]:
modelo.config.id2label = dict(enumerate(clases))
modelo.config.label2id = {c: i for i, c in enumerate(clases)}

repo = "mi-organizacion/clasificador-partes-siniestro"
modelo.push_to_hub(repo, private=True, commit_message="v1: F1 macro 0,91 en test")
tok.push_to_hub(repo, private=True)

# Cualquier compañero con acceso puede usarlo ya así:
from transformers import pipeline
clf = pipeline("text-classification", model=repo)
print(clf("Se ha roto la tubería del baño y ha inundado el piso de abajo"))


## Ejemplo 17. Detección de objetos

**Problema.** Un aparcamiento municipal quiere contar los coches en las fotos de sus cámaras para mostrar las plazas libres en un panel.

**Modelo.** `facebook/detr-resnet-50`, detector entrenado en COCO que localiza 80 clases de objetos, entre ellas `car`.

**Resultado esperado.** El número de coches y sus cajas delimitadoras.


In [ ]:
from PIL import Image
from transformers import pipeline

det = pipeline("object-detection", model="facebook/detr-resnet-50", revision="no_timm")  # sin depender de timm
foto = Image.open("camara_planta1.jpg")
coches = [d for d in det(foto) if d["label"] == "car" and d["score"] > 0.8]
AFORO = 120
print(f"Coches detectados: {len(coches)} | Plazas libres: {AFORO - len(coches)}")
for c in coches:
    print(c["box"])   # coordenadas xmin, ymin, xmax, ymax


## Ejemplo 18. Generación de imágenes con diffusers

**Problema.** Una agencia de marketing necesita bocetos rápidos para presentar ideas de campaña a un cliente antes de encargar la fotografía real.

**Modelo.** `stabilityai/sdxl-turbo`, modelo de difusión que genera una imagen en 1–4 pasos. Se distribuye bajo la Stability AI Community License: uso comercial gratuito para organizaciones con menos de 1 millón de dólares de ingresos anuales; por encima, exige licencia empresarial.

**Resultado esperado.** Una imagen de 512 × 512 píxeles en pocos segundos en una T4.


In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")
gen = torch.Generator("cuda").manual_seed(42)
imagen = pipe(
    prompt="botella de aceite de oliva sobre mesa de madera, luz de atardecer, fotografía de producto",
    num_inference_steps=2, guidance_scale=0.0, generator=gen,
).images[0]
imagen.save("boceto_campana.png")
